# Project B Lab：ML4Quant / LLM4Quant 的增量研究

这个实验把“模型看起来更聪明”拆成可验证的研究问题：在相同 label、时间切分、top-bottom 组合规则和成本下，文本特征是否超过 market-only baseline？

**顺序**：先跑 `mean`，再跑 `market_ridge`，然后看 lexical；semantic provider 是可选扩展，不是默认前提。先预测，再交易，再审计 coverage 和 as-of 时间。

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

PROJECT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src" / "data.py").exists() and (p / "src" / "evaluate.py").exists()
)
sys.path.insert(0, str(PROJECT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT / ".matplotlib"))

from src.run import main as run_baseline
from src.evaluate import top_bottom_returns

LAB_OUTPUT = PROJECT / "reports" / "lab_ml_outputs"
LAB_OUTPUT.mkdir(parents=True, exist_ok=True)
run_baseline(output_dir=LAB_OUTPUT)
metrics = pd.read_json(LAB_OUTPUT / "metrics.json").T
validation_metrics = pd.read_json(LAB_OUTPUT / "validation_metrics.json").T
print("test metrics")
display(metrics.round(4))
print("validation metrics")
display(validation_metrics.round(4))

## 1. 验证集和测试集必须分开

本项目的默认边界是 train 60% / validation 20% / test 20%。模型只在 train fit；validation 用于研究选择；test 用于最后一次报告。`predictions.csv` 有 `split` 列，因此可以检查每个模型在两个窗口的表现是否一致。

**TODO（运行前写下预期）**：如果 lexical 特征只是合成文本的粗粒度代理，validation 和 test 的 rank IC 是否应该同号？如果不同，先检查覆盖率和随机波动，再谈模型。

In [ ]:
import matplotlib.pyplot as plt
from src.data import make_panel, DataConfig

predictions = pd.read_csv(LAB_OUTPUT / "predictions.csv", parse_dates=["decision_date"])
fig, ax = plt.subplots(figsize=(11, 5))
for (split_name, model), group in predictions.groupby(["split", "model"]):
    if split_name != "test":
        continue
    portfolio = top_bottom_returns(group, cost_bps=10)
    equity = (1 + portfolio["net"]).cumprod()
    ax.plot(portfolio["decision_date"], equity, label=model)
ax.set_title("Project B: test-period top-bottom net equity (10 bps)")
ax.set_ylabel("equity")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(PROJECT / "reports" / "lab_ml_test_equity.png", dpi=150)
plt.show()

coverage = make_panel(DataConfig())
coverage["asof_ok"] = pd.to_datetime(coverage["published_at"]) < pd.to_datetime(coverage["decision_date"])
print({"rows": len(coverage), "asof_ok_rate": float(coverage["asof_ok"].mean()), "unique_texts": int(coverage["text"].nunique())})

## 2. 成本压力和模型增量

**TODO**：先预测哪个模型在 50 bps 成本下会最稳定，并说明你依据的是 IC、turnover 还是经济机制。下面固定 test 预测，只改变成本；这不是重新调参。

In [ ]:
rows = []
for (split_name, model), group in predictions.groupby(["split", "model"]):
    if split_name != "test":
        continue
    for cost_bps in [0, 10, 25, 50]:
        result = top_bottom_returns(group, cost_bps=cost_bps)
        rows.append({
            "model": model,
            "cost_bps": cost_bps,
            "average_net": float(result["net"].mean()),
            "average_turnover": float(result["turnover"].mean()),
        })
cost_table = pd.DataFrame(rows)
print(cost_table.pivot(index="model", columns="cost_bps", values="average_net").round(5))
fig, ax = plt.subplots(figsize=(10, 4))
for model, group in cost_table.groupby("model"):
    ax.plot(group["cost_bps"], group["average_net"], marker="o", label=model)
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Test average net spread under transaction costs", xlabel="cost (bps)", ylabel="average monthly net")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(PROJECT / "reports" / "lab_ml_cost_sensitivity.png", dpi=150)
plt.show()

## 3. 文本审计与交付检查

不要把 semantic score 当作收益概率，也不要只报告最好的 ablation。完成以下清单：

- [ ] 用 `published_at < decision_date` 检查 as-of 约束；
- [ ] 报告 train/validation/test 日期边界和每段样本数；
- [ ] 比较 mean、market-only、lexical 的 validation/test rank IC 和 net spread；
- [ ] 报告成本翻倍后的结论以及文本覆盖率；
- [ ] 若运行 semantic，记录模型名、cache、设备、截断和失败样本；
- [ ] 在 `reports/memo_template.md` 写出“文本没有增量”也成立的负结果。

面试表达：先说明 label 和发布时间，再说明 baseline，最后才说明文本/LLM 的边际贡献与失败边界。